In [1]:
import pandas as pd
import numpy as np
import gc
import io
import os
import csv
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
HEAD=180000
SEED=71

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils
import modules.encode as encode

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

import lightgbm as lgb

HEAD = 160000
SEED = 71

f001 = [
    'f001_NAME_CONTRACT_TYPE',
    'f001_CODE_GENDER',
    'f001_FLAG_OWN_CAR',
    'f001_FLAG_OWN_REALTY',
    'f001_NAME_TYPE_SUITE',
    'f001_NAME_INCOME_TYPE',
    'f001_NAME_EDUCATION_TYPE',
    'f001_NAME_FAMILY_STATUS',
    'f001_NAME_HOUSING_TYPE',
    'f001_OCCUPATION_TYPE',
    'f001_WEEKDAY_APPR_PROCESS_START',
    'f001_ORGANIZATION_TYPE',
    'f001_FONDKAPREMONT_MODE',
    'f001_HOUSETYPE_MODE',
    'f001_WALLSMATERIAL_MODE',
    'f001_EMERGENCYSTATE_MODE',
]

f002 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]

f003 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]


ALL_CAT = f001 + f002 + f003

In [2]:
high_var = "high_var_f1"
high_var_f0 = "high_var_f0"

In [3]:
import re
def read_feather_with_head(file_path):
    return pd.read_feather(file_path).head(HEAD)

def sanitize_feature_name(name):
    return re.sub(r"[+(),. ]", "_", name)

def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + sanitize_feature_name(line).strip() + ".f" for line in f]
        return features

### tunning nhẹ param vì auc_train và auc_valid đang có dấu hiệu overfit

In [4]:
# param tunning tạm ổn với n_features=850, record=160000
param = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6, # or 10 - 15
    'num_leaves': 63, # 63 or 20 - 50
    'max_bin': 255,
    'min_child_weight': 15, # 5 or 10 - 20
    'min_data_in_leaf': 400, # 100 - 200
    'reg_lambda': 1, #0.5 or 0.01 or 0.1 # L2 regularization term on weights.
    'reg_alpha': 1, # 0.5  # L1 regularization term on weights.
    'colsample_bytree': 0.7,
    'subsample': 0.6, # 0.5
    # 'nthread': 12,
    'bagging_freq': 1,
    'verbose': -1,
    'seed': SEED,
    # thêm cấu hình cho GPU
    'device_type': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

In [ ]:
n_thread=12 # cpu 6 cores 12 threads

feature_paths = read(ROOT + f"/.log/_used/{high_var}.txt") # lấy tất các feature có var chấp nhận được
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/lgbm_imp_{high_var}.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split', 'auc_train', 'auc_valid'])
    
    file_paths = read(ROOT + f"/.log/_used/{high_var_f0}.txt")
    futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
    chunk_dfs = [future.result() for future in as_completed(futures)]
    X_f0 = pd.concat(chunk_dfs, axis=1)
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        X = pd.concat([X, X_f0], axis=1)
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(1000), lgb.early_stopping(100)])

        importance_gain = np.round(model.feature_importance(importance_type='gain'), 5)
        importance_split = model.feature_importance(importance_type='split')

        y_pred_train = model.predict(X_train)
        y_pred_val = model.predict(X_val)
        
        auc_train = np.round(roc_auc_score(y_train, y_pred_train), 5)
        auc_val = np.round(roc_auc_score(y_val, y_pred_val), 5)

        feature_names = X.columns
        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i], auc_train, auc_val])
        csv_rows.sort(key=lambda x: x[2], reverse=True)
        writer.writerows(csv_rows)
        print(len(csv_rows))
        chunk+=1

### Mô hình phi tuyến nên phải xét cả những feature có tương quan thấp với target